<a href="https://colab.research.google.com/github/mohitagr18/LLMENGG/blob/main/Structured_LLM_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Structured LLM Engineering**

# Setup & Installs

In [ ]:
!pip install -q uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 29.6 MB/s eta 0:00:00


In [1]:
!uv pip install --system -q instructor outlines dspy-ai pydantic openai transformers torch

In [2]:
import time
import statistics
from pydantic import BaseModel, Field, field_validator
import instructor
from openai import OpenAI
import outlines
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import dspy


In [3]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("API Key set successfully!")

API Key set successfully!


# Instructor Demo

In [4]:
# --- Cell 1: Instructor Core Extraction ---
import instructor
from openai import OpenAI
from pydantic import BaseModel, Field

client = instructor.from_openai(OpenAI())

class ProductReview(BaseModel):
    product_name: str = Field(description="Name of the product being reviewed")
    rating: int = Field(description="Rating from 1 to 5 stars")
    pros: list[str] = Field(description="Positive aspects")
    cons: list[str] = Field(description="Negative aspects or complaints")
    would_recommend: bool = Field(description="Whether reviewer recommends product")

review_text = """
My two-week experience with the Sony WH-1000XM5 has revealed superior noise cancellation
and outstanding battery performance. Though the headphones are somewhat expensive and
 their carrying case is rather large, I confidently endorse them for individuals who
 travel frequently.
"""

review = client.chat.completions.create(
    model="gpt-4o-mini",
    response_model=ProductReview,
    messages=[{"role": "user", "content": f"Extract review data from: {review_text}"}],
)

print(f"Product: {review.product_name}")
print(f"Rating: {review.rating}/5")
print(f"Pros: {review.pros}")
print(f"Cons: {review.cons}")
print(f"Recommend: {review.would_recommend}")

Product: Sony WH-1000XM5
Rating: 5/5
Pros: ['superior noise cancellation', 'outstanding battery performance']
Cons: ['somewhat expensive', 'carrying case is rather large']
Recommend: True


In [5]:
# --- Cell 2: Instructor Validation & Retries ---
from pydantic import BaseModel, field_validator

class EmailAddress(BaseModel):
    address: str
    domain: str

    @field_validator("address")
    @classmethod
    def must_be_valid_email(cls, v: str) -> str:
        if "@" not in v:
            raise ValueError(f"'{v}' is not a valid email — must contain '@'")
        return v.lower()

    @field_validator("domain")
    @classmethod
    def must_match_address(cls, v: str, values) -> str:
        if "address" in values.data:
            expected = values.data["address"].split("@")[-1]
            if v != expected:
                raise ValueError(f"Domain '{v}' doesn't match email domain '{expected}'")
        return v

email_object = client.chat.completions.create(
    model="gpt-4o-mini",
    response_model=EmailAddress,
    max_retries=3,  # Intercepts ValidationError and re-prompts model
    messages=[{"role": "user", "content": "Parse: mohit@gmail.com (company email)"}],
)

print("Validated Email Object:", email_object)

Validated Email Object: address='mohit@gmail.com' domain='gmail.com'


In [6]:
# --- Cell 3: Instructor Real-Time Partial Streaming ---
from pydantic import BaseModel

class Order(BaseModel):
    customer: str
    items: list[str]
    total: float
    status: str

order_text = "Order from John Doe: 5x MacBook Pro, 2x AirPods Pro. Total: $4798. Status: confirmed."

for partial_order in client.chat.completions.create_partial(
    model="gpt-4o-mini",
    response_model=Order,
    messages=[{"role": "user", "content": f"Extract order: {order_text}"}],
):
    print(f"Customer: {partial_order.customer or '...'} | Total: {partial_order.total or '...'}")

Customer: ... | Total: ...
Customer: ... | Total: ...
Customer: ... | Total: ...
Customer: ... | Total: ...
Customer: John | Total: ...
Customer: John Doe | Total: ...
Customer: John Doe | Total: ...
Customer: John Doe | Total: ...
Customer: John Doe | Total: ...
Customer: John Doe | Total: ...
Customer: John Doe | Total: ...
Customer: John Doe | Total: ...
Customer: John Doe | Total: ...
Customer: John Doe | Total: ...
Customer: John Doe | Total: ...
Customer: John Doe | Total: ...
Customer: John Doe | Total: ...
Customer: John Doe | Total: ...
Customer: John Doe | Total: ...
Customer: John Doe | Total: ...
Customer: John Doe | Total: ...
Customer: John Doe | Total: ...
Customer: John Doe | Total: ...
Customer: John Doe | Total: ...
Customer: John Doe | Total: ...
Customer: John Doe | Total: ...
Customer: John Doe | Total: ...
Customer: John Doe | Total: ...
Customer: John Doe | Total: ...
Customer: John Doe | Total: ...
Customer: John Doe | Total: ...
Customer: John Doe | Total: ...


# Outlines Demo

In [7]:
# --- Cell 4: Outlines Regex & Choice Constraints ---
import outlines, torch
from outlines.types import Regex, Choice
from transformers import AutoModelForCausalLM, AutoTokenizer

LOCAL_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL)
hf_model = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL, torch_dtype=torch.float32)
local_model = outlines.from_transformers(hf_model, tokenizer)

# Guaranteed regex format
phone = local_model("Contact us at +1 800-123-4567", Regex(r"\+\d{1,3}-\d{5}-\d{5}"))

# Guaranteed choice enum
category = local_model("Classify ticket: 'My payment failed'", Choice(["billing", "technical", "shipping"]))

print("Phone Match:", phone)
print("Category Choice:", category)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=67) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=58) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Phone Match: +1-80012-34567
Category Choice: shipping


In [12]:
# --- Cell 5: Outlines Pydantic Schema-Guided Decoding ---
from typing import Literal
from pydantic import BaseModel, Field

class LineItem(BaseModel):
    description: str
    quantity: int = Field(gt=0)
    price: float = Field(gt=0)

class Invoice(BaseModel):
    invoice_number: str
    customer_name: str
    items: list[LineItem]
    payment_status: Literal["paid", "pending", "overdue"]

invoice_text = "Invoice #INV-2011-001 for ABC Corp. 1. Wireless Mouse - 5 units @ $25.00 each. Status: pending."

result_json = local_model(f"Extract invoice: {invoice_text}", Invoice, max_new_tokens=512)
invoice = Invoice.model_validate_json(result_json)

print(f"Invoice #: {invoice.invoice_number} | Customer: {invoice.customer_name} | Status: {invoice.payment_status}")

Invoice #: INV-2011-001 | Customer: ABC Corp. | Status: pending


# DSPy Demo

In [9]:
# --- Cell 7: DSPy Signatures & ReAct Agent ---
import dspy

lm = dspy.LM("openai/gpt-4o-mini")
dspy.configure(lm=lm)

# 1. Chain-of-Thought Module
class QuestionAnswering(dspy.Signature):
    """Answer questions with short factual responses."""
    question: str = dspy.InputField()
    answer: str = dspy.OutputField()

cot = dspy.ChainOfThought(QuestionAnswering)
res = cot(question="If a train travels 90 m/h for 2.5 hours, how far does it go?")
print("Reasoning:", res.reasoning)
print("Answer:", res.answer)

# 2. ReAct Agent Tool Loop
def search_web(query: str) -> str:
    return f"Search results for '{query}': US GDP 2025 = $30.7T, population = 330M"

def calculator(expression: str) -> str:
    return str(eval(expression))

agent = dspy.ReAct("question -> answer", tools=[search_web, calculator], max_iters=5)
agent_res = agent(question="What is the GDP per capita of US in 2025?")
print("Agent Answer:", agent_res.answer)

Reasoning: To find the distance traveled by the train, we can use the formula: 

Distance = Speed × Time.

Here, the speed of the train is 90 miles per hour (m/h) and the time the train travels is 2.5 hours. Now we can calculate the distance:

Distance = 90 m/h × 2.5 hours = 225 miles.
Answer: 225 miles
Agent Answer: $93,030.30


In [10]:
# --- Cell 8: DSPy Automated Few-Shot Compilation ---
from dspy.teleprompt import BootstrapFewShot

trainset = [
    dspy.Example(question="What is DSPy?", answer="DSPy is a framework for programming language models.").with_inputs("question"),
    dspy.Example(question="What is Pydantic?", answer="Pydantic is a data validation library for Python.").with_inputs("question"),
]

def answer_metric(example, pred, trace=None):
    expected = set(example.answer.lower().split())
    predicted = set(pred.answer.lower().split())
    return 1.0 if len(expected & predicted) / len(expected) >= 0.5 else 0.0

class SimpleQA(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate = dspy.ChainOfThought("question -> answer")

    def forward(self, question):
        return self.generate(question=question)

teleprompter = BootstrapFewShot(metric=answer_metric, max_bootstrapped_demos=2)
compiled_program = teleprompter.compile(SimpleQA(), trainset=trainset)

output = compiled_program(question="What is DSPy?")
print("Compiled Program Output:", output.answer)


100%|██████████| 2/2 [00:04<00:00,  2.31s/it]


Bootstrapped 2 full traces after 1 examples for up to 1 rounds, amounting to 2 attempts.
Compiled Program Output: DSPy is a Python library that streamlines the building and deployment of data science models by providing a framework for creating, validating, and managing data pipelines.


# Latency Benchmark Harness

In [ ]:
# -------------------------------------------------------------------
# Benchmark Harness Setup
# -------------------------------------------------------------------
def measure_latency(func, runs=5):
    """Executes a target function across multiple runs and returns latency stats."""
    latencies = []
    for _ in range(runs):
        start = time.perf_counter()
        result = func()
        end = time.perf_counter()
        latencies.append((end - start) * 1000)  # Convert to milliseconds
    return {
        "mean_ms": statistics.mean(latencies),
        "min_ms": min(latencies),
        "max_ms": max(latencies),
        "p95_ms": sorted(latencies)[int(0.95 * len(latencies))],
        "last_result": result
    }

print("Starting Latency Benchmark across Instructor, Outlines, and DSPy...\n")

Starting Latency Benchmark across Instructor, Outlines, and DSPy...



In [ ]:
# -------------------------------------------------------------------
# 1. Instructor Latency & Retry Benchmark
# -------------------------------------------------------------------
client = instructor.from_openai(OpenAI())

class ValidatedOrder(BaseModel):
    customer: str
    amount: float
    status: str

    @field_validator("amount")
    @classmethod
    def check_positive(cls, v):
        if v <= 0:
            raise ValueError("Amount must be strictly positive")
        return v

def run_instructor_success():
    return client.chat.completions.create(
        model="gpt-4o-mini",
        response_model=ValidatedOrder,
        messages=[{"role": "user", "content": "Extract order: Customer John, $150.00, status paid"}],
    )

instructor_stats = measure_latency(run_instructor_success, runs=5)
print(f"[Instructor - First-Pass Success] Mean Latency: {instructor_stats['mean_ms']:.2f} ms | Max: {instructor_stats['max_ms']:.2f} ms")



[Instructor - First-Pass Success] Mean Latency: 1253.83 ms | Max: 2042.51 ms


In [ ]:
# -------------------------------------------------------------------
# 2. Outlines FSM Compilation vs. Guided Generation Latency
# -------------------------------------------------------------------
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
hf_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32)

# Measure FSM Compilation Time (Upfront Overhead)
compile_start = time.perf_counter()
local_model = outlines.from_transformers(hf_model, tokenizer)
compile_end = time.perf_counter()
fsm_compile_time_ms = (compile_end - compile_start) * 1000

def run_outlines_guided():
    return local_model(
        "Extract date from 'Order placed on 2024-09-20':",
        outlines.types.Regex(r"\d{4}-\d{2}-\d{2}")
    )

outlines_stats = measure_latency(run_outlines_guided, runs=5)
print(f"[Outlines - FSM Compilation Delay]: {fsm_compile_time_ms:.2f} ms (Upfront Startup)")
print(f"[Outlines - Guided Token Generation]: Mean Latency: {outlines_stats['mean_ms']:.2f} ms | P95: {outlines_stats['p95_ms']:.2f} ms")


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=68) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=68) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


[Outlines - FSM Compilation Delay]: 168.36 ms (Upfront Startup)
[Outlines - Guided Token Generation]: Mean Latency: 14497.87 ms | P95: 50722.94 ms


In [ ]:
# -------------------------------------------------------------------
# 3. DSPy Module Execution Latency
# -------------------------------------------------------------------
lm = dspy.LM("openai/gpt-4o-mini")
dspy.configure(lm=lm)

class QA(dspy.Signature):
    question: str = dspy.InputField()
    answer: str = dspy.OutputField()

cot_module = dspy.ChainOfThought(QA)

def run_dspy_cot():
    return cot_module(question="What is 15 * 12?")

dspy_stats = measure_latency(run_dspy_cot, runs=5)
print(f"[DSPy - ChainOfThought Module]: Mean Latency: {dspy_stats['mean_ms']:.2f} ms | P95: {dspy_stats['p95_ms']:.2f} ms\n")


[DSPy - ChainOfThought Module]: Mean Latency: 1671.52 ms | P95: 7729.26 ms



In [ ]:
# -------------------------------------------------------------------
# Comparative Benchmark Results Table
# -------------------------------------------------------------------
print("=" * 70)
print(f"{'Framework':<20} | {'Phase':<22} | {'Mean Latency (ms)':<15}")
print("=" * 70)
print(f"{'Instructor':<20} | {'API Response (Success)':<22} | {instructor_stats['mean_ms']:<15.2f}")
print(f"{'Outlines':<20} | {'FSM Compilation (Once)':<22} | {fsm_compile_time_ms:<15.2f}")
print(f"{'Outlines':<20} | {'Guided Decoding (Local)':<22} | {outlines_stats['mean_ms']:<15.2f}")
print(f"{'DSPy':<20} | {'ChainOfThought Module':<22} | {dspy_stats['mean_ms']:<15.2f}")
print("=" * 70)

Framework            | Phase                  | Mean Latency (ms)
Instructor           | API Response (Success) | 1253.83        
Outlines             | FSM Compilation (Once) | 168.36         
Outlines             | Guided Decoding (Local) | 14497.87       
DSPy                 | ChainOfThought Module  | 1671.52        
